
# Collaborative Filtering — Theory

**Recommender Systems — SCIA — Session 03**

This notebook is a *reading* companion to the slides: theory only, no code to
write. The hands-on implementation is in `03_CollaborativeFiltering_TP.ipynb`.

## Outline
- [1. From content to collaboration](#1)
- [2. Notation](#2)
- [3. The two vectors](#3)
- [4. The utility matrix](#4)
- [5. Predicting a rating](#5)
- [6. The cost function](#6)
- [7. Training: a custom loop](#7)
- [8. From parameters to recommendations](#8)



<a name="1"></a>
## 1. From content to collaboration

Until now, we used item *features* to predict how a user would rate an item
— that's content-based filtering, and it's genuinely useful. But it needs
good metadata, and it can only ever recommend things similar to what a user
already likes.

Collaborative filtering takes a different route: it ignores what an item
*is* and only looks at who rated what. The idea is to learn about users and
items purely from the pattern of ratings. Users who rated the same items
similarly are "close" to each other; items rated similarly by the same
users are "close" to each other too.

### Advantages
- No domain knowledge necessary — no genres, no tags, no descriptions.
- Can help a user discover something a content-based system would never
  suggest, because *similar users* liked it, not because it looks similar
  to what the user already knows.
- Only needs the rating matrix — no feature engineering.

### Disadvantages
- **Cold start**: a brand-new user or a brand-new item has no ratings yet,
  so the model has nothing to learn from.
- Blending in extra features later (to fix cold start) can get messy.



<a name="2"></a>
## 2. Notation

| Symbol | Meaning |
|---|---|
| $r(i,j)$ | $1$ if user $j$ rated movie $i$, $0$ otherwise |
| $y(i,j)$ | rating given by user $j$ to movie $i$ (defined only if $r(i,j)=1$) |
| $w^{(j)}$ | parameter vector for user $j$ |
| $b^{(j)}$ | bias scalar for user $j$ |
| $x^{(i)}$ | feature vector for movie $i$ |
| $n_u$ | number of users |
| $n_m$ | number of movies |
| $n$ | number of latent features |
| $X$ | matrix stacking every $x^{(i)}$ — shape $(n_m, n)$ |
| $W$ | matrix stacking every $w^{(j)}$ — shape $(n_u, n)$ |
| $b$ | vector stacking every $b^{(j)}$ — shape $(n_u,)$ |
| $R$ | matrix of every $r(i,j)$ — shape $(n_m, n_u)$ |



<a name="3"></a>
## 3. The two vectors

A collaborative filtering model learns two things simultaneously:

- for every **user**, a parameter vector $w$ that embodies their taste;
- for every **movie**, a feature vector $x$ of the same length.

The dot product of the two, plus a bias term, estimates the rating that
user would give that movie:

$$ \hat{y}^{(i,j)} = w^{(j)} \cdot x^{(i)} + b^{(j)} $$

Neither vector is handed to the model — both are *learned* from the
observed ratings. The feature vector for a movie must work reasonably well
for every user who rated it; the parameter vector for a user must work
reasonably well for every movie they rated. That mutual constraint is where
the name comes from: every user's ratings collaborate to shape every
movie's representation, and vice versa.



<a name="4"></a>
## 4. The utility matrix

Ratings live in a matrix $Y$ (movies &times; users): 0.5 to 5 in 0.5 steps,
undefined wherever nobody rated. $R$ is the same shape and marks which
entries of $Y$ are real observations ($R(i,j)=1$) versus unobserved
($R(i,j)=0$).

|          | User 1 | User 2 | User 3 |
|----------|:---:|:---:|:---:|
| Movie A  | 5   | —   | 4   |
| Movie B  | —   | 4   | —   |
| Movie C  | 3   | —   | 5   |

Every "—" is a cell where $R=0$: the model must never be penalised for its
prediction there, since there's no ground truth to compare against. Getting
this mask wrong is the single most common bug in a from-scratch
implementation — more on that in the TP.



<a name="5"></a>
## 5. Predicting a rating

Once $X$, $W$ and $b$ are learned, predicting an unobserved rating is just
the dot product from section 3: $\hat{y}^{(i,j)} = w^{(j)} \cdot x^{(i)} +
b^{(j)}$. Computed for every $(i,j)$ pair at once, this is a single matrix
multiplication — the same operation whether the cell was observed during
training or not, which is exactly what lets the model fill in the blanks.



<a name="6"></a>
## 6. The cost function

The collaborative filtering cost function is:

$$
J(X, W, b) = \frac{1}{2} \sum_{(i,j):\, r(i,j)=1} \left( w^{(j)} \cdot x^{(i)} + b^{(j)} - y^{(i,j)} \right)^2
+ \underbrace{\frac{\lambda}{2} \sum_{j} \sum_{k} \left(w^{(j)}_k\right)^2 + \frac{\lambda}{2} \sum_{i} \sum_{k} \left(x^{(i)}_k\right)^2}_{\text{regularisation}}
$$

The first sum runs "for all $i$, $j$ where $r(i,j)=1$" — equivalently, over
*every* pair, multiplied by $r(i,j)$ so unobserved pairs contribute nothing:

$$
= \frac{1}{2} \sum_{j} \sum_{i} r(i,j) \cdot \left( w^{(j)} \cdot x^{(i)} + b^{(j)} - y^{(i,j)} \right)^2
+ \text{regularisation}
$$

$\lambda$ controls how strongly the learned vectors are pulled toward zero
— without it, the model can drive the cost arbitrarily low by growing $X$
and $W$ without bound, badly overfitting a matrix that's usually very
sparse.

A vectorised implementation (using matrix multiplication instead of nested
loops) is essential in practice, since the cost is recomputed at every
training step. The linear algebra itself isn't the focus of this course —
what matters pedagogically is realising that the `R` mask must be applied
*before* squaring the error, exactly as in the loop version above. Skipping
it is an easy mistake with a vectorised formula, and it silently corrupts
the whole training run.



<a name="7"></a>
## 7. Training: a custom loop

$X$, $W$ and $b$ are *all* free parameters being learned at once — there's
no fixed input feeding a stack of layers, so the usual
`model.fit()` pattern doesn't apply. Instead, the training loop is written
by hand:

- repeat until convergence:
    - compute the forward pass (the cost, given current $X$, $W$, $b$)
    - compute the gradient of the cost with respect to each parameter
    - update each parameter using the learning rate and its gradient

TensorFlow's `tf.GradientTape` handles the differentiation automatically:
operations performed on `tf.Variable`s inside a `with tf.GradientTape() as
tape:` block are recorded, and `tape.gradient(cost, [X, W, b])` returns the
gradient of `cost` with respect to each of the three parameter tensors. An
optimiser (e.g. Adam) then applies the update.



<a name="8"></a>
## 8. From parameters to recommendations

Once training converges, predicting every rating for every user is one
matrix multiplication: $\hat{Y} = XW^\top + b$. For a given user, sorting
their column of $\hat{Y}$ in descending order and dropping the movies they
already rated gives a ranked recommendation list.

In practice, raw predicted scores for the top hundred or so movies often
sit in a narrow range — barely distinguishable. A common refinement is to
additionally filter by a minimum number of ratings (avoiding
recommending obscure movies on thin evidence) and sort the survivors by
their average rating, combining the model's personalised signal with a
simple popularity prior.

---
Next: open `03_CollaborativeFiltering_TP.ipynb` to implement this.
